# 04 指定したGLBにTokenRigで骨を入れ、Takaの動作を移す

ColabのL4 GPUを選びます。入力はリグ付け前の静的GLB 1本です。
01と同じ設定のTokenRigで骨を入れ、02と同じTakaの5動作を移します。

In [ ]:
from google.colab import drive, output
from pathlib import Path
import sys

drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/Colab Notebooks/GameCreation/MotionStudio/10_ImageToCharacter' #@param {type:'string'}
ROOT = Path(PROJECT_DIR).expanduser()
if not (ROOT / 'tools/colab_runtime.py').is_file():
    raise FileNotFoundError('GitHubのコード一式があるフォルダーをPROJECT_DIRに指定してください。')
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / 'tools'))
import importlib, colab_runtime
importlib.reload(colab_runtime)
from colab_runtime import (
    check_project, create_job, run_logged,
    prepare_image, prepare_rig, prepare_web, serve_web, save_result,
)
ROOT = check_project(ROOT)
JOB = create_job(ROOT)
print('コードの読み込み先:', ROOT)
print('今回の作業先:', JOB)

required_04 = ('tools/rig_to_taka.py', 'tools/orient_pixal_input.py',
               'tools/pixal3d_refine/photo_paint.py')
missing_04 = [name for name in required_04 if not (ROOT / name).is_file()]
if missing_04:
    raise FileNotFoundError('04のソースが不足しています。最新版ソースZIPをDriveに展開してください: '
                            + ', '.join(missing_04))

In [ ]:
ENGINE = Path('/content/motionstudio10/tokenrig-engine')
run_logged([sys.executable, ROOT / 'reference/setup_l4.py', '--base', ENGINE,
            '--rig-only'], JOB / 'setup.log')

## GLBを1本指定する

Drive上のパスを `GLB_PATH` に設定します。
03の `model.refined.glb` なら `FROM_PIXAL3D_03` をオンにします。
他のツールで作った正面が+ZのGLBはオフにします。
向きや変形は、出力されたビューアで確認してください。

In [ ]:
from rig_to_taka import run_case

GLB_PATH = '' #@param {type:'string'}
FROM_PIXAL3D_03 = True #@param {type:'boolean'}
if not GLB_PATH.strip():
    raise ValueError('処理する静的GLBを1本指定してください。')
JOB = create_job(ROOT)
TRANSFER = run_case(ROOT, ENGINE, Path(GLB_PATH).expanduser(), JOB, FROM_PIXAL3D_03)
ARCHIVE = save_result(ROOT, TRANSFER, JOB)

In [ ]:
WEB = prepare_web(ROOT, TRANSFER, JOB)
if 'AVATAR_SERVER' in globals():
    AVATAR_SERVER.shutdown()
    AVATAR_SERVER.server_close()
AVATAR_SERVER = serve_web(WEB)
output.serve_kernel_port_as_iframe(
    AVATAR_SERVER.server_address[1], height=850, cache_in_notebook=False,
)